# Hybrid LSTM+GAT Fraud Detection — IBM Dataset
Standalone companion to *Isolating Graph Topology from Model Architecture in
GNN-Based Fraud Detection* (Amiri & Jaf, 2026). Extends the paper's
fixed-architecture / variable-topology framework with three architecture
variants, to test whether the paper's topology ranking
(`multi_relation` > `hybrid` > `intra_group`) holds when the *architecture*
changes instead of the topology.

| File | Architecture | Node granularity |
|---|---|---|
| `ibm/lstm_gat_sequential_model.py` | LSTM → GAT pipeline | Transaction |
| `ibm/lstm_gat_parallel_model.py` | LSTM ‖ GAT, cross-attention fusion | Transaction |
| `ibm/account_gat_homogeneous_model.py` | Unchanged GATv2, new topology | Account |

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Update `REPO_URL` and `IBM_DATA_PATH` in Cell 2 below
3. Run cells top to bottom

---
## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
import torch

torch_version = torch.__version__.split('+')[0]
cuda_version  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_version}  |  CUDA: {cuda_version}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'pandas', 'matplotlib', 'pyarrow'], check=True)
print('Done.')

---
## Cell 2 — Clone this repo and set the dataset path

In [ ]:
import os

REPO_URL = 'https://github.com/<your-username>/hybrid-gnn-lstm-fraud.git'  # ← update
IBM_DATA_PATH = '/content/reduced_dataset.parquet'  # ← update (upload, or mount Drive below)

if not os.path.isdir('/content/hybrid-gnn-lstm-fraud'):
    !git clone -q {REPO_URL} /content/hybrid-gnn-lstm-fraud

os.makedirs('/content/outcomes', exist_ok=True)
print('Repo ready at /content/hybrid-gnn-lstm-fraud')

---
## Cell 2b — (Optional) Mount Google Drive
Skip if you're uploading the dataset directly or it's already on the Colab disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'  # ← update and re-run Cell 2 if needed

---
## Cell 3 — Load and preprocess the IBM dataset
Shared by all runs below (loaded once, reused).

In [ ]:
import os, sys

os.chdir('/content/hybrid-gnn-lstm-fraud/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg = ibm_cfg.IBMFraudConfig()
cfg.OUTCOME_DIR = '/content/outcomes'

import utils as ibm_utils

try:
    df_ibm
except NameError:
    df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg)

print(f'{len(df_ibm):,} transactions loaded.')

---
## Cell 4 — GATv2 baseline (fixed architecture, for comparison)
Runs all 3 graph strategies (multi_relation, hybrid, intra_group).
Expected time: ~25–40 min on T4 GPU.

In [ ]:
import gatv2_model

gatv2_results = gatv2_model.run_all_strategies(df_ibm, cfg)
print('\nGATv2 baseline done.')

---
## Cell 5 — Sequential LSTM→GAT
Expected time: ~35–55 min on T4 GPU.

In [ ]:
import lstm_gat_sequential_model as lstm_seq

lstm_seq_results = lstm_seq.run_all_strategies(df_ibm, cfg)
print('\nSequential LSTM→GAT done.')

---
## Cell 6 — Parallel LSTM‖GAT
Expected time: ~45–70 min on T4 GPU (dual-branch, slowest of the three).

In [ ]:
import lstm_gat_parallel_model as lstm_par

lstm_par_results = lstm_par.run_all_strategies(df_ibm, cfg)
print('\nParallel LSTM‖GAT done.')

---
## Cell 7 — Homogeneous Account-Level GAT
Different node granularity: aggregates transactions into one node per
account, then reuses the unchanged `GATFraudModel` over three
account-level graph strategies. Expected time: ~5–10 min.

In [ ]:
import account_gat_homogeneous_model as acct_gat

acct_cfg = acct_gat.AccountFraudConfig()
acct_cfg.OUTCOME_DIR = '/content/outcomes'

df_accounts = acct_gat.build_account_dataframe(df_ibm, acct_cfg)
acct_results = acct_gat.run_all_strategies(df_accounts, acct_cfg)
print('\nHomogeneous Account-Level GAT done.')

---
## Cell 8 — Combined comparison across all architectures

In [ ]:
import pandas as pd

all_results = {}
all_results.update(gatv2_results)
all_results.update(lstm_seq_results)
all_results.update(lstm_par_results)
all_results.update(acct_results)

rows = []
for key, res in all_results.items():
    m = res['test_metrics']
    rows.append({
        'run': key,
        'model_arch': res.get('model_arch', ''),
        'graph_strategy': res.get('graph_strategy', ''),
        'f1': round(m['f1'], 4), 'prec': round(m['prec'], 4), 'rec': round(m['rec'], 4),
        'auc': round(m['auc'], 4), 'ap': round(m['ap'], 4),
    })

df_summary = pd.DataFrame(rows).sort_values(['model_arch', 'f1'], ascending=[True, False])
df_summary.to_csv('/content/outcomes/ibm_all_architectures_summary.csv', index=False)
df_summary

---
## Cell 9 — Download results

In [ ]:
!cd /content/outcomes && zip -qr /content/outcomes.zip .
from google.colab import files
files.download('/content/outcomes.zip')